In [ ]:
import pandas as pd
import numpy as np
import toad

# ==========================================
# 1. 构造测试数据
# ==========================================
np.random.seed(42)

def create_data(target_name, size, bad_rate_base):
    # 模拟概率 prob: 0 到 1 之间
    prob = np.random.beta(2, 5, size) 
    # 模拟标签 label: 概率越高，label=1 的可能性越大
    label = (np.random.rand(size) < (prob * bad_rate_base)).astype(int)
    return pd.DataFrame({'prob': prob, 'label': label, 'target': target_name})

# 构造 train, valid, oot 三个集合
df_train = create_data('train', 1000, 0.5)
df_valid = create_data('valid', 500, 0.45)
df_oot   = create_data('oot', 500, 0.4)

# 合并成一个总表
df = pd.concat([df_train, df_valid, df_oot], ignore_index=True)

print("--- 原始数据抽样 ---")
print(df.head())
print("-" * 30)

# ==========================================
# 2. 使用 Combiner 进行分箱
# ==========================================

# 初始化分箱器
combiner = toad.transform.Combiner()

# 【核心】仅在 train 数据集上进行训练 (Fit)
# n_bins=10 表示 10 等频分箱，method='quantile'
combiner.fit(df[df['target'] == 'train'][['prob', 'label']], y='label', method='quantile', n_bins=10)

# 查看分箱边界 (看看 train 是怎么切分的)
print("分箱边界 (Rules):")
print(combiner.export())
print("-" * 30)

# ==========================================
# 3. 将分箱规则应用到全量数据 (Transform)
# ==========================================
# transform 会把原来的连续值 prob 转换成离散的箱号 (0, 1, 2...)
df_binned = combiner.transform(df)


# ==========================================
# 4. 统计样本数与正样本数 (Label=1)
# ==========================================

# 按照 target 和 分箱号(prob) 分组统计
# 注意：transform 后，原来的 'prob' 列变成了箱号索引
stats = df_binned.groupby(['target', 'prob']).agg(
    total_count=('label', 'count'),
    pos_count=('label', 'sum')
).reset_index()

# 为了更直观，我们将结果透视，对比 train/valid/oot
final_report = stats.pivot(
    index='prob', 
    columns='target', 
    values=['total_count', 'pos_count']
).fillna(0).astype(int)

# 重新排序列的顺序，让 train 在前面
final_report = final_report.reindex(columns=['train', 'valid', 'oot'], level=1)

print("最终统计报表 (基于 Train 分箱):")
print(final_report)

# ==========================================
# 5. 验证：查看每组的坏样本率 (Bad Rate) 
# ==========================================
# 这一步是为了验证分箱的单调性
bad_rate_report = (final_report['pos_count'] / final_report['total_count']).round(4)
print("\n各组 Bad Rate 验证:")
print(bad_rate_report)

--- 原始数据抽样 ---
       prob  label target
0  0.353677      0  train
1  0.248558      0  train
2  0.415959      0  train
3  0.159968      0  train
4  0.550283      1  train
------------------------------
分箱边界 (Rules):
{'prob': [0.09620149023212136, 0.1404257305486772, 0.18415975611631608, 0.22497660108410283, 0.26267124424236, 0.3096936137238047, 0.3588801660282713, 0.42767201520463444, 0.506897828152197]}
------------------------------
最终统计报表 (基于 Train 分箱):
       total_count           pos_count          
target       train valid oot     train valid oot
prob                                            
0              100    54  46         4     0   1
1              100    40  49         4     3   2
2              100    42  56        10     1   3
3              100    46  49        14     5   8
4              100    40  47        12     3   7
5              100    56  55        12     8   2
6              100    60  43        13     4   9
7              100    49  67         9    11   8


In [2]:
import pandas as pd
import numpy as np
import toad

# ==========================================
# 1. 模拟数据生成 (与之前一致)
# ==========================================
np.random.seed(42)
def create_data(target_name, size, bad_rate_base):
    prob = np.random.beta(2, 5, size)
    label = (np.random.rand(size) < (prob * bad_rate_base)).astype(int)
    return pd.DataFrame({'prob': prob, 'label': label, 'target': target_name})

df = pd.concat([
    create_data('train', 1000, 0.5),
    create_data('valid', 500, 0.45),
    create_data('oot', 500, 0.4)
], ignore_index=True)

# ==========================================
# 2. Toad 分箱处理
# ==========================================
combiner = toad.transform.Combiner()
# 在 train 上 fit
combiner.fit(df[df['target'] == 'train'][['prob', 'label']], y='label', method='quantile', n_bins=10)

# 转换全量数据
df_binned = combiner.transform(df)

# ==========================================
# 3. 提取分箱区间名称 (Intervals)
# ==========================================
# combiner.export()['prob'] 返回的是切分点列表 [0.05, 0.12, ...]
# 我们需要将其转换为可读的区间字符串
splits = combiner.export()['prob']
# 补全边界以生成区间
bins = [-np.inf] + splits + [np.inf]
interval_names = [f"[{bins[i]:.4f}, {bins[i+1]:.4f})" for i in range(len(bins)-1)]

# 创建一个映射表 (箱号 -> 区间名)
bin_map = {i: name for i, name in enumerate(interval_names)}

# ==========================================
# 4. 生成三张结果表
# ==========================================
results = {}
for target in ['train', 'valid', 'oot']:
    # 筛选对应数据
    temp_df = df_binned[df_binned['target'] == target]
    
    # 聚合统计
    stats = temp_df.groupby('prob').agg(
        总样本数=('label', 'count'),
        正样本数=('label', 'sum')
    ).reset_index()
    
    # 映射区间名并清理格式
    stats['分箱区间'] = stats['prob'].map(bin_map)
    
    # 调整列顺序并只保留你需要的列
    final_table = stats[['分箱区间', '总样本数', '正样本数']]
    results[target] = final_table

# ==========================================
# 5. 打印结果
# ==========================================
for name, table in results.items():
    print(f"\n--- {name.upper()} 统计表 ---")
    print(table)

# 你也可以通过 results['train'] 获取对应的 DataFrame


--- TRAIN 统计表 ---
               分箱区间  总样本数  正样本数
0    [-inf, 0.0962)   100     4
1  [0.0962, 0.1404)   100     4
2  [0.1404, 0.1842)   100    10
3  [0.1842, 0.2250)   100    14
4  [0.2250, 0.2627)   100    12
5  [0.2627, 0.3097)   100    12
6  [0.3097, 0.3589)   100    13
7  [0.3589, 0.4277)   100     9
8  [0.4277, 0.5069)   100    23
9     [0.5069, inf)   100    29

--- VALID 统计表 ---
               分箱区间  总样本数  正样本数
0    [-inf, 0.0962)    54     0
1  [0.0962, 0.1404)    40     3
2  [0.1404, 0.1842)    42     1
3  [0.1842, 0.2250)    46     5
4  [0.2250, 0.2627)    40     3
5  [0.2627, 0.3097)    56     8
6  [0.3097, 0.3589)    60     4
7  [0.3589, 0.4277)    49    11
8  [0.4277, 0.5069)    39    10
9     [0.5069, inf)    74    21

--- OOT 统计表 ---
               分箱区间  总样本数  正样本数
0    [-inf, 0.0962)    46     1
1  [0.0962, 0.1404)    49     2
2  [0.1404, 0.1842)    56     3
3  [0.1842, 0.2250)    49     8
4  [0.2250, 0.2627)    47     7
5  [0.2627, 0.3097)    55     2
6  [0.3097, 0.358

In [9]:
import pandas as pd
import numpy as np
import re
import toad

# ==========================================
# 1. 构造示例数据 (Mock Data) - 修正为宽表结构
# ==========================================
np.random.seed(42)
n_samples = 1000

# df_data 现在是宽表，每个变量是一列
df_data = pd.DataFrame({
    'TZ_1381_m1': np.random.randn(n_samples) * 10 + 10,
    'TZ_0272_m1': np.random.randn(n_samples) * 5 + 5,
    'target': np.random.choice(['train', 'valid', 'oot'], p=[0.6, 0.2, 0.2], size=n_samples),
    'label': np.random.choice([0, 1], p=[0.9, 0.1], size=n_samples)
})

# 分箱规则表 df_bins 保持不变
df_bins = pd.DataFrame({
    'variable': ['TZ_1381_m1']*4 + ['TZ_0272_m1']*3,
    'value': [0, 1, 2, 3, 0, 1, 2], 
    'bin': ['[-inf, -0.5)', '[-0.5, 2.5)', '[2.5, 4.5)', '[4.5, inf)', 
            '[-inf, 1.5)', '[1.5, 5.5)', '[5.5, inf)']
})

# ==========================================
# 2. 核心处理逻辑 (适配宽表)
# ==========================================
def get_inner_splits(bin_series):
    """从分箱字符串中提取内部分割点"""
    bps = set()
    for b in bin_series:
        nums = re.findall(r'-?\d+\.?\d*', str(b).replace('-inf', '').replace('inf', ''))
        for n in nums:
            if n: bps.add(float(n))
    return sorted(list(bps))

def calculate_wide_stats_for_wide_table(df_data, df_bins):
    all_results = []
    variables = df_bins['variable'].unique()
    
    for var in variables:
        # 如果 df_data 里没有这个特征列，则跳过
        if var not in df_data.columns:
            continue
            
        var_bins = df_bins[df_bins['variable'] == var].sort_values('value').copy()
        splits = get_inner_splits(var_bins['bin'])
        
        # 构建该变量的基础宽表框架
        base_df = var_bins[['variable', 'bin']].rename(columns={'variable': 'var_names', 'bin': '分箱'})
        base_df['value_key'] = var_bins['value'] 
        
        # 遍历三种数据集类型
        for dataset_type in ['train', 'valid', 'oot']:
            # 按数据集拆分
            df_split = df_data[df_data['target'] == dataset_type]
            
            if df_split.empty:
                for col in ['样本数', '样本占比', '逾期率', 'IV', 'lift', '累积lift']:
                    base_df[f'{dataset_type}_{col}'] = np.nan
                continue
            
            # 借助 toad 计算基础指标：注意这里直接传入宽表的特征列 df_split[var]
            ks_df = toad.metrics.KS_bucket(
                df_split[var], 
                df_split['label'], 
                bucket=splits, 
                method='step'
            )
            ks_df['value_key'] = range(len(ks_df))
            
            # --- 计算分箱 IV ---
            goods = ks_df['total'] - ks_df['bads']
            good_dist = goods / goods.sum() if goods.sum() > 0 else 0
            bad_dist = ks_df['bads'] / ks_df['bads'].sum() if ks_df['bads'].sum() > 0 else 0
            
            # 平滑处理
            good_dist_safe = np.where(good_dist == 0, 0.0001, good_dist)
            bad_dist_safe = np.where(bad_dist == 0, 0.0001, bad_dist)
            
            ks_df['IV'] = (good_dist_safe - bad_dist_safe) * np.log(good_dist_safe / bad_dist_safe)
            
            # 提取需要的列 (IV 在 lift 之前)
            metrics_df = ks_df[['value_key', 'total', 'total_prop', 'bad_rate', 'IV', 'lift', 'cum_lift']].copy()
            
            # 加上 dataset_type 前缀
            rename_dict = {
                'total': f'{dataset_type}_样本数',
                'total_prop': f'{dataset_type}_样本占比',
                'bad_rate': f'{dataset_type}_逾期率',
                'IV': f'{dataset_type}_IV',
                'lift': f'{dataset_type}_lift',
                'cum_lift': f'{dataset_type}_累积lift'
            }
            metrics_df.rename(columns=rename_dict, inplace=True)
            
            # 拼接到基础宽表上
            base_df = pd.merge(base_df, metrics_df, on='value_key', how='left')
            
            # 填补空箱的样本数和占比
            base_df[f'{dataset_type}_样本数'] = base_df[f'{dataset_type}_样本数'].fillna(0)
            base_df[f'{dataset_type}_样本占比'] = base_df[f'{dataset_type}_样本占比'].fillna(0)
            
        # 删掉用于连接的索引列
        base_df.drop(columns=['value_key'], inplace=True)
        all_results.append(base_df)
        
    return pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

# ==========================================
# 3. 运行并查看结果
# ==========================================
result_df = calculate_wide_stats_for_wide_table(df_data, df_bins)

pd.set_option('display.max_columns', None)  # 方便完整查看宽表列
print(result_df.head(10))

    var_names            分箱  train_样本数  train_样本占比  train_逾期率  train_IV  \
0  TZ_1381_m1  [-inf, -0.5)         89    0.150847   0.089888  0.010820   
1  TZ_1381_m1   [-0.5, 2.5)         49    0.083051   0.000000  0.606982   
2  TZ_1381_m1    [2.5, 4.5)         46    0.077966   0.021739  0.072188   
3  TZ_1381_m1    [4.5, inf)        406    0.688136   0.081281  0.015079   
4  TZ_0272_m1   [-inf, 1.5)        130    0.220339   0.084615  0.008385   
5  TZ_0272_m1    [1.5, 5.5)        181    0.306780   0.077348  0.002564   
6  TZ_0272_m1    [5.5, inf)        279    0.472881   0.060932  0.012213   

   train_lift  train_累积lift  valid_样本数  valid_样本占比  valid_逾期率  valid_IV  \
0    1.262707      1.000000         24    0.111111   0.166667  0.023440   
1    0.000000      0.953331         17    0.078704   0.058824  0.032354   
2    0.305383      1.056679          8    0.037037   0.250000  0.045418   
3    1.141802      1.141802        167    0.773148   0.107784  0.004819   
4    1.188645      1.188

In [8]:
result_df

,var_names,分箱,train_样本数,train_样本占比,train_逾期率,train_IV,train_lift,train_累积lift,valid_样本数,valid_样本占比,valid_逾期率,valid_IV,valid_lift,valid_累积lift,oot_样本数,oot_样本占比,oot_逾期率,oot_IV,oot_lift,oot_累积lift
0,TZ_1381_m1,"[-inf, -0.5)",48,0.162162,0.125000,0.009953,1.233333,1.233333,6,0.061224,0.333333,0.355040,4.083333,4.083333,11,0.114583,0.000000,0.939182,0.000000,1.000000
1,TZ_1381_m1,"[-0.5, 2.5)",23,0.077703,0.130435,0.007076,1.286957,1.250704,6,0.061224,0.000000,0.432836,0.000000,2.041667,8,0.083333,0.000000,0.652549,0.000000,1.129412
2,TZ_1381_m1,"[2.5, 4.5)",20,0.067568,0.050000,0.029034,0.493333,1.084249,4,0.040816,0.000000,0.270360,0.000000,1.531250,2,0.020833,0.000000,0.129754,0.000000,1.246753
3,TZ_1381_m1,"[4.5, inf)",205,0.692568,0.097561,0.001220,0.962602,1.000000,82,0.836735,0.073171,0.011202,0.896341,1.000000,75,0.781250,0.160000,0.071921,1.280000,1.280000
4,TZ_0272_m1,"[-inf, 1.5)",73,0.244966,0.109589,0.070887,1.555121,1.555121,15,0.145631,0.066667,0.055373,0.528205,1.000000,17,0.155963,0.176471,0.235260,2.404412,2.404412
5,TZ_0272_m1,"[1.5, 5.5)",41,0.137584,0.048780,0.017809,0.692218,1.244779,12,0.116505,0.000000,0.958672,0.000000,1.080420,20,0.183486,0.100000,0.024295,1.362500,1.841216
6,TZ_0272_m1,"[5.5, inf)",184,0.617450,0.059783,0.017720,0.848344,1.000000,76,0.737864,0.157895,0.055298,1.251012,1.251012,72,0.660550,0.041667,0.184844,0.567708,1.000000
